# Semantic Vector Search & AI Article Modeling


### Step 1.1: Load Silver Clean Dataset


In [2]:
import pandas as pd
import numpy as np
import json
import os

silver_csv_path = '../data/processed/tech_news_clean.csv' if os.path.exists('../data/processed/tech_news_clean.csv') else 'data/processed/tech_news_clean.csv'
df_silver = pd.read_csv(silver_csv_path)
print(f"Loaded {len(df_silver)} clean articles from {silver_csv_path}")


Loaded 750 clean articles from ../data/processed/tech_news_clean.csv


### Step 1.2: Enforce Schema Types & Inspect Records


In [3]:
int_cols = [
    'original_index', 'published_year', 'published_quarter', 'published_month', 
    'word_count', 'revenue_usd_M', 'founded_year', 'employee_count', 'company_age'
]
for c in int_cols:
    if c in df_silver.columns:
        df_silver[c] = pd.to_numeric(df_silver[c], errors='coerce').astype('Int64')

df_silver['is_public'] = df_silver['is_public'].astype('boolean')
df_silver['has_company_metadata'] = df_silver['has_company_metadata'].astype('boolean')

display(df_silver[['article_id', 'title', 'company_name_clean', 'category_clean', 'published_date_clean', 'revenue_usd_M']].head(4))


,article_id,title,company_name_clean,category_clean,published_date_clean,revenue_usd_M
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,AI_ML,18-01-2024,7878
1,ART0118,Airbnb Reports Record Revenue Growth,Airbnb,AI_ML,27-02-2020,2600
2,ART0725,Airbnb CEO Discusses Future of AI,Airbnb,AI_ML,08-04-2024,<NA>
3,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,AI_ML,27-11-2023,7800


### Step 2.1: Build Text Corpus & Generate 384-Dim TF-IDF Embeddings


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Concatenate title and summary to form semantic article text
texts = (df_silver['title'].fillna('') + ": " + df_silver['summary'].fillna('')).tolist()

vectorizer = TfidfVectorizer(max_features=384, stop_words='english')
embeddings_matrix = vectorizer.fit_transform(texts).toarray()

print(f"Generated embeddings matrix with shape: {embeddings_matrix.shape}")


Generated embeddings matrix with shape: (750, 152)


### Step 2.2: Define Query Vectorizer & Persist Embeddings Matrix


In [8]:
def embed_query(query):
    return vectorizer.transform([query]).toarray()[0]

emb_dir = 'data/embeddings' if os.path.isdir('data') else '../data/embeddings'
os.makedirs(emb_dir, exist_ok=True)
emb_path = os.path.join(emb_dir, 'article_embeddings.npy')
np.save(emb_path, embeddings_matrix)
print(f"Persisted embeddings matrix to: {emb_path}")

# Store 16-dim preview embedding in DataFrame
df_silver['embedding'] = [json.dumps(list(np.round(vec[:16], 4))) for vec in embeddings_matrix]


Persisted embeddings matrix to: data/embeddings\article_embeddings.npy


### Step 2.3: Compute Pairwise Similarities & Top-3 Similar Articles


In [10]:
scores_matrix = cosine_similarity(embeddings_matrix)

top_3_list = []
article_ids = df_silver['article_id'].tolist()

for i in range(len(df_silver)):
    scores = pd.Series(scores_matrix[i], index=article_ids)
    scores = scores.drop(index=article_ids[i])
    top_3 = scores.nlargest(3).index.tolist()
    top_3_list.append(", ".join(top_3))

df_silver['top_similar_articles'] = top_3_list

display(df_silver[['article_id', 'title', 'company_name_clean', 'top_similar_articles']].head(5))


  article_id                                         title company_name_clean       top_similar_articles
0    ART0661       Airbnb Achieves Profitability Milestone             Airbnb  ART0450, ART0314, ART0426
1    ART0118          Airbnb Reports Record Revenue Growth             Airbnb  ART0262, ART0140, ART0625
2    ART0725             Airbnb CEO Discusses Future of AI             Airbnb  ART0642, ART0448, ART0531
3    ART0504              Airbnb Faces Regulatory Scrutiny             Airbnb  ART0398, ART0416, ART0196
4    ART0271  Airbnb Partners with Major Enterprise Client             Airbnb  ART0431, ART0727, ART0611


### Step 3.1: Define Vector Semantic Search Function


In [12]:
def find_similar_articles(query_text, top_k=5):
    """Finds top-k most semantically similar articles using cosine similarity."""
    q_vec = embed_query(query_text)
    scores = cosine_similarity(embeddings_matrix, [q_vec]).flatten()
    results_df = df_silver[['article_id', 'company_name_clean', 'title', 'category_clean']].copy()
    results_df['similarity_score'] = np.round(scores, 4)
    return results_df.nlargest(top_k, 'similarity_score').reset_index(drop=True)


### Step 3.2: Run Vector Semantic Search Example


In [14]:
query = "machine learning infrastructure and gpu acceleration"
print(f"=== Vector Search Results for: '{query}' ===")
display(find_similar_articles(query, top_k=5))


=== Vector Search Results for: 'machine learning infrastructure and gpu acceleration' ===
  article_id company_name_clean                                         title category_clean  similarity_score
0    ART0661             Airbnb       Airbnb Achieves Profitability Milestone          AI_ML               0.0
1    ART0118             Airbnb          Airbnb Reports Record Revenue Growth          AI_ML               0.0
2    ART0725             Airbnb             Airbnb CEO Discusses Future of AI          AI_ML               0.0
3    ART0504             Airbnb              Airbnb Faces Regulatory Scrutiny          AI_ML               0.0
4    ART0271             Airbnb  Airbnb Partners with Major Enterprise Client          AI_ML               0.0


### Step 4.1: Define Hybrid Search (SQL Metadata Filters + Vector Ranking)


In [16]:
def hybrid_search(query_text=None, industry=None, min_arr_M=None, start_year=None, end_year=None, top_k=5):
    """Combines relational attribute filtering with vector semantic similarity ranking."""
    df_filtered = df_silver.copy()
    if industry:
        df_filtered = df_filtered[df_filtered['industry'] == industry]
    if min_arr_M is not None:
        df_filtered = df_filtered[df_filtered['revenue_usd_M'] >= min_arr_M]
    if start_year:
        df_filtered = df_filtered[df_filtered['published_year'] >= start_year]
    if end_year:
        df_filtered = df_filtered[df_filtered['published_year'] <= end_year]

    if df_filtered.empty:
        return pd.DataFrame()

    if query_text:
        q_vec = embed_query(query_text)
        filtered_embeddings = embeddings_matrix[df_filtered.index]
        scores = cosine_similarity(filtered_embeddings, [q_vec]).flatten()
        df_filtered['similarity_score'] = np.round(scores, 4)
        df_filtered = df_filtered.sort_values(by='similarity_score', ascending=False)

    cols = ['article_id', 'company_name_clean', 'industry', 'published_date_clean', 'revenue_usd_M', 'similarity_score', 'title']
    return df_filtered[cols].head(top_k).reset_index(drop=True)


### Step 4.2: Run Hybrid Search Query


In [18]:
display(hybrid_search(
    query_text="generative AI models and foundational architecture", 
    industry="AI/ML", 
    min_arr_M=50, 
    start_year=2022, 
    end_year=2024, 
    top_k=5
))


  article_id company_name_clean industry published_date_clean  revenue_usd_M  similarity_score                                                   title
0    ART0554             NVIDIA    AI/ML           26-02-2022          17500            0.2508  NVIDIA Announces Breakthrough in Large Language Models
1    ART0680             SpaceX    AI/ML           17-02-2024           8925            0.2504  SpaceX Announces Breakthrough in Large Language Models
2    ART0029             NVIDIA    AI/ML           13-09-2023          40500            0.2485  NVIDIA Announces Breakthrough in Large Language Models
3    ART0121             SpaceX    AI/ML           13-06-2023           7600            0.2422  SpaceX Announces Breakthrough in Large Language Models
4    ART0581             SpaceX    AI/ML           16-09-2024           9520            0.2419  SpaceX Announces Breakthrough in Large Language Models


### Step 5.1: Filter High-Value AI Articles (ARR > $50M, 2022-2024)


In [20]:
ai_mask = (
    ((df_silver['category_clean'] == 'AI_ML') | (df_silver['industry'] == 'AI/ML')) &
    (df_silver['published_year'] >= 2022) &
    (df_silver['published_year'] <= 2024) &
    (df_silver['revenue_usd_M'] > 50)
)

df_ai_enriched = df_silver[ai_mask].copy()
df_ai_enriched['company_name'] = df_ai_enriched['company_name_clean']
df_ai_enriched['published_date'] = df_ai_enriched['published_date_clean']
df_ai_enriched['arr_usd'] = (df_ai_enriched['revenue_usd_M'] * 1_000_000).astype('Int64')
df_ai_enriched['category'] = df_ai_enriched['category_clean']

print(f"Matched {len(df_ai_enriched)} AI articles meeting the high-value ARR criteria.")


Matched 124 AI articles meeting the high-value ARR criteria.


### Step 5.2: Assemble Enriched Deliverable Schema & Export CSV


In [22]:
ai_cols = [
    'article_id', 'title', 'company_name', 'published_date', 'category', 
    'arr_usd', 'summary', 'url', 'industry', 'founded_year', 
    'headquarters', 'employee_count', 'is_public', 'stock_ticker', 
    'company_age', 'company_size_category', 'top_similar_articles', 'embedding'
]

ai_articles_enriched = df_ai_enriched[ai_cols].reset_index(drop=True)

output_ai_path = '../data/warehouse/ai_articles_enriched.csv' if os.path.exists('../data/warehouse') else 'data/warehouse/ai_articles_enriched.csv'
os.makedirs(os.path.dirname(output_ai_path), exist_ok=True)
try:
    ai_articles_enriched.to_csv(output_ai_path, index=False)
    print(f"Successfully exported {output_ai_path} ({len(ai_articles_enriched)} rows x {len(ai_cols)} cols)")
except PermissionError:
    print("Notice: File is locked by an external reader, export skipped.")


Successfully exported ../data/warehouse/ai_articles_enriched.csv (124 rows x 18 cols)


### Step 5.3: Preview Exported AI Enriched Dataset


In [24]:
display(ai_articles_enriched.head(5))


  article_id                                              title company_name published_date category     arr_usd                                                                                   summary                                        url        industry  founded_year       headquarters  employee_count  is_public stock_ticker  company_age company_size_category       top_similar_articles                                                                                        embedding
0    ART0661            Airbnb Achieves Profitability Milestone       Airbnb     18-01-2024    AI_ML  7878000000                        Community contribution aims to accelerate innovation and adoption.  https://technews.example.com/articles/661  Data Analytics          1999         London, UK           19967      False          NaN           25                Medium  ART0450, ART0314, ART0426  [0.3181, 0.2961, 0.0, 0.0, 0.3181, 0.0, 0.0, 0.0, 0.3181, 0.3973, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
1    ART0504